# Day 3 Project — Solution: Local AI CLI Assistant

This notebook shows the complete solution. Read it **after** you've built your own version.

## Part 1: The `ask()` Helper Function

In [ ]:
import ollama

MODEL = "llama3.2"

def ask(question: str, system: str = "") -> str:
    """Send a question to the local model and return the reply."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})
    response = ollama.chat(model=MODEL, messages=messages)
    return response["message"]["content"]


print(ask("What is the capital of Japan?"))

## Part 2: Demo Mode

In [ ]:
SYSTEM = "You are a friendly, knowledgeable assistant. Keep answers concise — 2-3 sentences."

DEMO_QUESTIONS = [
    "What is machine learning in one sentence?",
    "What is the difference between RAM and storage?",
    "Give me a Python tip I might not know.",
]

for q in DEMO_QUESTIONS:
    print(f"You: {q}")
    reply = ask(q, SYSTEM)
    print(f"AI: {reply}\n")

## Part 3: Multi-Turn CLI Loop

The code below uses `input()`, which requires a real terminal — it cannot run inside a notebook kernel. **Save it as `assistant.py`** and run `python assistant.py` in your terminal.

```python
import ollama

MODEL  = "llama3.2"
SYSTEM = "You are a helpful assistant. Be concise and clear."

messages = [{"role": "system", "content": SYSTEM}]

print("Local AI CLI — type 'exit' to quit\n")

while True:
    question = input("You: ").strip()

    if not question:
        continue

    if question.lower() in ("exit", "quit", "q"):
        print("Goodbye!")
        break

    messages.append({"role": "user", "content": question})

    try:
        response = ollama.chat(model=MODEL, messages=messages)
        reply    = response["message"]["content"]
        messages.append({"role": "assistant", "content": reply})
        print(f"AI: {reply}\n")
    except Exception as e:
        print(f"Error: {e}\n")
        print("  → Is Ollama running?  macOS: open the app · Linux/Windows: ollama serve")
        print("  → Is the model pulled?  ollama pull llama3.2\n")
```

## How It Works

**`messages` list as memory:** The model has no memory between calls — it only sees what's in the `messages` list. We start with the system prompt and append each user/assistant turn. Growing this list is what makes the model "remember" previous turns.

**`while True:` loop:** Runs forever until `break` is called. Standard Python pattern for interactive programs — you don't know in advance how many questions the user will ask.

**`continue` for empty input:** If the user presses Enter without typing anything, `continue` skips the model call and returns to `input()`. Without this, an empty string would be sent to the model.

**Error handling:** Wrapping the model call in `try/except` prevents an Ollama crash (e.g., model not found, server stopped) from crashing the whole program. The user gets a helpful message and the loop keeps running.

**Customise it:** Change `SYSTEM` to give the model any persona — a coding tutor, a Socratic teacher, a pirate, a minimalist who answers in one sentence. The system prompt is the primary tool for shaping behaviour.